# Rishton Ke Mod (`रिश्तों के मोड़`) — Automated Live-Action Video Generator & Assembler

Run this notebook in **Google Colab (Free T4 GPU)** or **Kaggle Notebooks (Free 2× T4 GPU, 30 hrs/week)**!
- **`PROVIDER = 'ltx'`** — **100% FREE, NO API KEY NEEDED** (`Lightricks/LTX-Video` open-source I2V on free T4 GPU)
- **`PROVIDER = 'veo' / 'fal' / 'sora'`** — Paid API mode for higher-res acting & lip-sync
- **`PROVIDER = 'none'`** — Assemble `.mp4` clips you dropped into Drive `clips/`

In [ ]:
# 1. Mount Google Drive (if on Colab) & install dependencies
import os
from pathlib import Path

IN_COLAB = os.path.exists('/content')
WORK_ROOT = Path('/content' if IN_COLAB else '/kaggle/working')
REPO_DIR = WORK_ROOT / 'contentcreation'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_EPISODE = Path('/content/drive/MyDrive/Rishton_Ke_Mod/2026-11-11')
    CLIPS_DIR = DRIVE_EPISODE / '02_Live_Action_Cinema_Kit' / 'clips'
else:
    DRIVE_EPISODE = WORK_ROOT / 'output'
    CLIPS_DIR = WORK_ROOT / 'clips'
CLIPS_DIR.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    !git clone https://github.com/codepranadev-svg/contentcreation.git {REPO_DIR}
%cd {REPO_DIR}
!git pull --ff-only
!pip -q install -r requirements.txt diffusers transformers accelerate sentencepiece imageio[ffmpeg] google-genai fal-client openai
print('Clips folder:', CLIPS_DIR)

In [ ]:
# 2. Generate live-action .mp4 clips ('ltx' is 100% FREE with NO API key on Colab/Kaggle T4 GPU)
from getpass import getpass

PROVIDER = 'ltx'  # 'ltx' (FREE T4 GPU, no key), 'veo', 'fal', 'sora', or 'none'

if PROVIDER == 'veo' and not os.environ.get('GEMINI_API_KEY'):
    os.environ['GEMINI_API_KEY'] = getpass('Enter GEMINI_API_KEY (for Veo 3.1): ')
elif PROVIDER == 'fal' and not os.environ.get('FAL_KEY'):
    os.environ['FAL_KEY'] = getpass('Enter FAL_KEY (for Kling 2.6 + LipSync): ')
elif PROVIDER == 'sora' and not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('Enter OPENAI_API_KEY (for Sora 2): ')

!python -m studio.live_action_api --episode-dir hindi/2026-11-11 --clips-dir "{CLIPS_DIR}" --provider {PROVIDER} --res 1080

In [ ]:
# 3. Save finished 1080p videos to Google Drive (Colab) or /kaggle/working/output (Kaggle)
import shutil
ep_dir = REPO_DIR / 'hindi/2026-11-11'
out_dir = DRIVE_EPISODE / '03_Colab_1080p_Output' if IN_COLAB else DRIVE_EPISODE
out_dir.mkdir(parents=True, exist_ok=True)

for fname in (
    'baarish-ki-woh-aakhiri-raat_full_16x9.mp4',
    'baarish-ki-woh-aakhiri-raat_shorts_9x16.mp4',
    'thumbnail_16x9.jpg',
    'thumbnail_9x16.jpg',
    'subtitles.srt',
):
    src = ep_dir / fname
    if src.exists():
        shutil.copy2(src, out_dir / fname)
print('Saved 1080p outputs to:', out_dir)